# SynnoDB: From DuckDB to Bespoke in One Import

SynnoDB is a drop-in replacement for DuckDB that transparently accelerates your SQL queries
with auto-generated bespoke C++ engines - while falling back to DuckDB for everything else.
No schema changes. No query rewrites. One import.

This notebook walks through the full journey for **TPC-H Q1-Q5**:

1. **Baseline** - run Q1-Q5 on vanilla DuckDB at SF20, 10 parameter instantiations each.
2. **Generate** - point SynnoDB at a `queries.json` file and let it write the engine.
3. **Drop in** - replace one import, re-run the identical queries, compare the numbers.

> **Prerequisites** - see [`docs/TUTORIAL_base_implementation.md`](../docs/TUTORIAL_base_implementation.md)
> for installation, TPC-H data generation, and model endpoint setup.

## Setup

Adjust the paths below if your data lives elsewhere.

In [1]:
import os, json, time, statistics
from pathlib import Path

DATA_ROOT = Path(os.environ.get("SYNNO_DATA_DIR", "/mnt/labstore/learneddb/synno_data"))
PARQUET_DIR = Path(
    os.environ.get("SYNNO_TPCH_PARQUET", DATA_ROOT / "workloads/tpch/tpch_parquet")
)
ENGINES_DIR = Path(os.environ.get("SYNNO_ENGINES_DIR", DATA_ROOT / "engines"))
MODEL       = os.environ.get("SYNNO_MODEL", "openai/unsloth/MiniMax-M3") # e.g. "anthropic/claude-sonnet-4-6", "gpt-5.4", "openai/unsloth/MiniMax-M3"
SF          = 20
TABLES      = ["customer", "lineitem", "nation", "orders", "part",
               "partsupp", "region", "supplier"]

ENGINES_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("SYNNO_ENGINES_DIR", str(ENGINES_DIR))
print("Parquet root :", PARQUET_DIR)
print("Engines dir  :", ENGINES_DIR)
print("Model        :", MODEL)

Parquet root : /mnt/labstore/learneddb/synno_data/workloads/tpch/tpch_parquet
Engines dir  : /mnt/labstore/learneddb/synno_data/engines
Model        : openai/unsloth/MiniMax-M3


---
## Step 1 - DuckDB Baseline

We run **Q1-Q5 on vanilla DuckDB** at SF20: 10 instantiations per query
(different placeholder values, drawn from the actual data), total wall-clock time recorded.
These exact SQL strings will be reused in Step 3 so the comparison is apples-to-apples.

### Register the BYO workload

The workload is described by a single self-describing JSON file. Each entry carries its SQL
template **and** a typed **spec** for each `[PLACEHOLDER]` slot - declaring its value *space*,
which is sampled at run time. A scalar placeholder is an `int`/`float`/`date`/`categorical`
spec; correlated or distinct placeholders share a `param_groups` spec:

```json
"6": {
  "sql": "... l_discount between [DISCOUNT] - 0.01 ... l_quantity < [QUANTITY] ...",
  "params": {
    "DATE":     { "type": "date",  "min": "1993-01-01", "max": "1997-01-01" },
    "DISCOUNT": { "type": "float", "min": 0.02, "max": 0.09, "step": 0.01 },
    "QUANTITY": { "type": "int",   "min": 24, "max": 25 }
  }
},
"7": {
  "sql": "... n1.n_name = '[NATION1]' ... n2.n_name = '[NATION2]' ...",
  "param_groups": [
    { "type": "sample", "placeholders": ["NATION1", "NATION2"], "domain": ["GERMANY", "CHINA", ...], "distinct": true }
  ]
}
```

`register_workload_from_json` reads it and derives the schema from the parquet via DuckDB.
Each placeholder's spec is sampled with the run's seeded RNG (a range → a uniform draw, a
`categorical` → a choice, a group → one joint row), so correlated placeholders stay aligned.
The typed spec is exactly what a BI dashboard renders as a slider (`int`/`float`), a dropdown
(`categorical`), or a date-picker (`date`).

In [2]:
import random
from synnodb.workloads.byo_workload import register_workload_from_json

QUERIES_JSON = Path("queries.json")  # lives next to this notebook

spec = register_workload_from_json(
    name="tpch_byo",
    queries_json=QUERIES_JSON,
    parquet_dir=PARQUET_DIR,
    scale_factors=(1, 2, SF),
    schema_example_table="lineitem",
)

print("Workload :", spec.name)
print("Tables   :", spec.tables)
print("Queries  :", spec.all_query_ids)

Workload : tpch_byo
Tables   : ('customer', 'lineitem', 'nation', 'orders', 'part', 'partsupp', 'region', 'supplier')
Queries  : ('1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22')


Here is what the queries look like - SQL templates with `[PLACEHOLDER]` slots, plus the typed
specs that define each slot's value space:

In [3]:
queries = json.loads(QUERIES_JSON.read_text())
for qid, entry in queries.items():
    print(f"=== Q{qid} ===")
    print(entry["sql"][:240], "...")
    print("params      :", entry.get("params", {}))
    print("param_groups:", entry.get("param_groups", []))
    print()

=== Q1 ===
select 
    l_returnflag,  
    l_linestatus,  
    sum(l_quantity) as sum_qty, 
    sum(l_extendedprice) as sum_base_price, 
    sum(l_extendedprice*(1-l_discount)) as sum_disc_price, 
    sum(l_extendedprice*(1-l_discount)*(1+l_tax)) as s ...
params      : {'DELTA': {'type': 'int', 'min': 60, 'max': 120, 'step': 1}}
param_groups: []

=== Q2 ===
select
    s_acctbal,
    s_name,
    n_name,
    p_partkey,
    p_mfgr,
    s_address,
    s_phone,
    s_comment
from
    part,
    supplier,
    partsupp,
    nation,
    region
where
    p_partkey = ps_partkey
    and s_suppkey = ps_sup ...
params      : {'SIZE': {'type': 'int', 'min': 1, 'max': 50, 'step': 1}, 'TYPE': {'type': 'categorical', 'values': ['TIN', 'NICKEL', 'BRASS', 'STEEL', 'COPPER']}, 'REGION': {'type': 'categorical', 'values': ['AFRICA', 'AMERICA', 'ASIA', 'EUROPE', 'MIDDLE EAST']}}
param_groups: []

=== Q3 ===
select l_orderkey,  
    sum(l_extendedprice*(1-l_discount)) as revenue, 
    o_orderdate,  
    o_ship

### Draw parameter instantiations

`query_gen_factory` fills the templates by sampling each placeholder's spec. We draw with a
fixed seed so the instantiations are **identical** for the DuckDB and SynnoDB runs.

In [4]:
N_REPS = 10
rng = random.Random(42)
gen = spec.query_gen_factory(None)

# gen(query_name, rng) -> (query_name, sql_with_values, params_dict)
instantiations = {}
for qid in spec.all_query_ids:
    instantiations[qid] = [gen(f"Q{qid}", rng) for _ in range(N_REPS)]

print(f"Drew {N_REPS} instantiations per query.")
for qid, insts in instantiations.items():
    sample_params = [i[2] for i in insts[:2]]
    print(f"  Q{qid}: {sample_params} ...")

Drew 10 instantiations per query.
  Q1: [{'DELTA': '100'}, {'DELTA': '67'}] ...
  Q2: [{'SIZE': '44', 'TYPE': 'COPPER', 'REGION': 'AFRICA'}, {'SIZE': '38', 'TYPE': 'STEEL', 'REGION': 'AFRICA'}] ...
  Q3: [{'SEGMENT': 'FURNITURE', 'DATE': '1995-03-04'}, {'SEGMENT': 'AUTOMOBILE', 'DATE': '1995-03-13'}] ...
  Q4: [{'DATE': '1997-08-26'}, {'DATE': '1996-07-11'}] ...
  Q5: [{'REGION': 'AMERICA', 'DATE': '1994-08-16'}, {'REGION': 'AFRICA', 'DATE': '1994-04-22'}] ...
  Q6: [{'DATE': '1994-05-17', 'DISCOUNT': '0.04', 'QUANTITY': '25'}, {'DATE': '1995-02-17', 'DISCOUNT': '0.06', 'QUANTITY': '24'}] ...
  Q7: [{'NATION1': 'CHINA', 'NATION2': 'JAPAN'}, {'NATION1': 'IRAQ', 'NATION2': 'GERMANY'}] ...
  Q8: [{'NATION': 'KENYA', 'REGION': 'MIDDLE EAST', 'TYPE': 'MEDIUM PLATED COPPER'}, {'NATION': 'PERU', 'REGION': 'AFRICA', 'TYPE': 'SMALL ANODIZED COPPER'}] ...
  Q9: [{'COLOR': 'puff'}, {'COLOR': 'almond'}] ...
  Q10: [{'DATE': '1993-04-01'}, {'DATE': '1993-10-05'}] ...
  Q11: [{'NATION': 'VIETNAM', '

### Run on DuckDB

In [5]:
import duckdb
from tqdm import tqdm

sf_dir = PARQUET_DIR / f"sf{SF}"

duck = duckdb.connect(":memory:")
duck.execute("PRAGMA disable_progress_bar")
for t in TABLES:
    duck.execute(
        f"CREATE VIEW {t} AS SELECT * FROM read_parquet('{sf_dir}/{t}.parquet')"
    )

baseline_times = {}
for qid, insts in tqdm(instantiations.items(), desc="Running baseline queries"):
    times = []
    for _, sql, _ in insts:
        t0 = time.perf_counter()
        duck.execute(sql).fetchall()
        times.append((time.perf_counter() - t0) * 1_000)
    baseline_times[qid] = times

duck.close()

total_duck = sum(sum(v)/len(v) for v in baseline_times.values())
print(f"{'Query':<8} {'Avg (ms)':>12} {'Median (ms)':>14}")
print("-" * 38)
for qid in spec.all_query_ids:
    t = baseline_times[qid]
    print(f"Q{qid:<7} {sum(t)/len(t):>12.1f} {statistics.median(t):>14.1f}")
print("-" * 38)
print(f"{'TOTAL':<8} {total_duck:>12.1f}")

Running baseline queries:   0%|          | 0/22 [00:00<?, ?it/s]

Running baseline queries: 100%|██████████| 22/22 [01:03<00:00,  2.88s/it]

Query        Avg (ms)    Median (ms)
--------------------------------------
Q1              123.5          120.1
Q2              129.2          130.2
Q3              519.8          520.3
Q4              128.1          126.1
Q5              177.5          172.7
Q6               65.8           66.4
Q7              114.0          112.4
Q8              145.9          144.9
Q9              312.1          309.6
Q10            2786.1         2777.7
Q11              52.0           49.6
Q12             112.0          110.0
Q13             140.1          135.9
Q14              96.4           97.3
Q15              99.3           99.2
Q16             147.8          146.6
Q17             134.0          133.1
Q18             265.0          267.6
Q19             108.4          107.2
Q20             136.3          137.2
Q21             396.6          394.7
Q22             142.2          148.8
--------------------------------------
TOTAL          6331.9


---
## Step 2 - Generate the SynnoDB Engine

You hand SynnoDB the same `queries.json` and a scale factor. It:

1. **Creates a storage plan** - decides how each query's columns are laid out in memory.
2. **Implements the engine** - writes single-threaded C++, compiles it, validates every output
   against DuckDB, then **auto-publishes** the binary into `ENGINES_DIR`.

This is a one-time cost. Once published the engine is discovered automatically across sessions.

### Start the SynnoDB engine

Constructing the `SynnoDB` driver spawns an in-process **live-UI dashboard** and prints its
URL (e.g. `http://localhost:8765`). Open it in a browser to watch generation unfold in real
time - input tokens, code size, per-query speedups, cost/runtime, and an activity log, all
refreshing every few seconds.

Every stage you chain on this same `db` - storage plan → base implementation →
`runOptimLoop` → `addMultiThreading` → `checkSfCorrectness` - streams onto **one continuous
timeline**, so the whole journey shows up on a single page instead of resetting per stage.
The dashboard stays up for the lifetime of this kernel; the URL is also available as
`db.dashboard_url`.


In [6]:
from synnodb import SynnoDB

db   = SynnoDB(workload="tpch_byo", model=MODEL, db_storage="in_memory", queries="1-5", data_dir=DATA_ROOT)

📊 SynnoDB live dashboard: http://localhost:8766


### Storage plan

In [7]:
plan = db.createStoragePlan()

print("Run :", plan.run_id)
print()
print(plan.text[:600], "...")

2026-06-30 15:24:33 INFO:synnodb.main:Using database source: DBStorage.IN_MEMORY. Disk DB dir: None
2026-06-30 15:24:33 INFO:synnodb.cpp_runner.prepare_repo.prepare_workspace:Writing 0 read-only artifact files to `/home/jwehrstein/SynnoDB/tutorials/output` for benchmark tpch_byo
2026-06-30 15:24:33 INFO:synnodb.cpp_runner.prepare_repo.prepare_workspace:Restoring prepared repo from cache: 4598feea6272b216a299c1d44512ee827606f1ba0d4df4b170690c82ec9aceb6.pkl
2026-06-30 15:24:33 INFO:synnodb.main:Workspace root: output
2026-06-30 15:24:33 INFO:synnodb.utils.model_setup:Using LiteLLM model: openai/unsloth/MiniMax-M3 (provider: openai)
2026-06-30 15:24:33 INFO:synnodb.utils.model_setup:No LLM_API_BASE set, defaulting to local model endpoint: http://dgx02:13506/v1
2026-06-30 15:24:33 WARNING:synnodb.conversations.conversation:Auto-U mode enabled: automatically proceeding with all prompts without asking for user confirmation. Make sure this is what you want!


2026-06-30 15:24:33 INFO:synnodb.conversations.conversation:==================== Prompt ====================
2026-06-30 15:24:33 INFO:synnodb.conversations.conversation:Your task is to analyze the workload and produce a creative in-memory storage-layout summary for the tables accessed by the query. You have the flexibility to return detailed, free-form text that explores not only conventional storage-layout recommendations but also unconventional, novel, and even 'crazy' storage designs.
You are encouraged to include additional ideas, new partitioning strategies, speculative encoding techniques, or experimental ways of grouping and organizing columns or data.
For each accessed table, feel free to be inventive and elaborate on possibilities such as hybrid layouts, speculative SoA/AoS (Array of Structures/Structure of Arrays) approaches, novel column encodings, or adaptive partitioning.
Use this as an opportunity to push beyond current norms and propose storage techniques that might be f

### Base implementation

We feed the plan **content** straight in via `storage_plan=plan.text`, so this step needs
no W&B. If you instead chain off a logged storage-plan run, pass its run id with
`db.createBaseImpl(storage_plan_wandb_id=plan.run_id)`. Provide exactly one of the two.

In [9]:
impl = db.createBaseImpl(storage_plan=plan.text)

print("Workspace :", impl.workspace)
print("Files     :", sorted(impl.files))
print()
print(f"Engine published to: {ENGINES_DIR}")

2026-06-30 15:11:51 DEBUG:asyncio:Using selector: EpollSelector
2026-06-30 15:11:51 INFO:synnodb.main:Using database source: DBStorage.IN_MEMORY. Disk DB dir: None
2026-06-30 15:11:51 DEBUG:synnodb.synth_framework.git_snapshotter:Fetching snapshots from cache repo 'cache_repo'...
2026-06-30 15:11:51 DEBUG:synnodb.synth_framework.git_snapshotter:Restored snapshot: 8dea4df1b1e6c5bf848ef9901d778e24d2197d11
2026-06-30 15:11:51 INFO:synnodb.cpp_runner.prepare_repo.prepare_workspace:Writing 5 read-only artifact files to `/home/jwehrstein/SynnoDB/tutorials/output` for benchmark tpch_byo
2026-06-30 15:11:51 INFO:synnodb.cpp_runner.prepare_repo.prepare_workspace:Writing 5 artifact files (args_parser.hpp, query_impl.hpp, query_impl.cpp, parquet_reader.hpp, parquet_reader.cpp) to `/home/jwehrstein/SynnoDB/tutorials/output` for optim
2026-06-30 15:11:51 INFO:synnodb.cpp_runner.prepare_repo.prepare_workspace:Restoring prepared repo from cache: b4ed08610720419cb5fcb8e8bf924302c9e1a397ee487ad24ac42be


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.



InternalServerError: litellm.InternalServerError: InternalServerError: OpenAIException - Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

---
## Step 3 - Drop In SynnoDB

The only change is **one import line** and two extra keyword arguments to `connect()`:

```diff
- import duckdb
+ import synnodb as duckdb
+ from synnodb.router import RouterMode, RouterPolicy

  con = duckdb.connect(
      ":memory:",
+     engines=str(ENGINES_DIR),
+     policy=RouterPolicy(mode=RouterMode.SAMPLED, cross_check_rate=1.0),
  )
```

Every other line - the view setup, the `execute()` calls, `fetchall()` - is identical.

### Open the drop-in connection

In [ ]:
import synnodb as duckdb  # <- only change
from synnodb.router import RouterMode, RouterPolicy

con = duckdb.connect(
    ":memory:",
    engines=str(ENGINES_DIR),
    policy=RouterPolicy(mode=RouterMode.SAMPLED, cross_check_rate=1.0),
)

sf_dir = PARQUET_DIR / f"sf{SF}"
for t in TABLES:
    con.duckdb.execute(
        f"CREATE VIEW {t} AS SELECT * FROM read_parquet('{sf_dir}/{t}.parquet')"
    )

con.refresh_engines()
n = con.router_stats()["registry"]["templates"]
print(f"Discovered {n} engine template(s) under {ENGINES_DIR}")

### Run the same queries with the same parameter values

We iterate over `instantiations` - the exact SQL strings from Step 1.

In [ ]:
synno_times = {}
for qid, insts in instantiations.items():
    times = []
    for _, sql, _ in insts:
        t0 = time.perf_counter()
        con.execute(sql).fetchall()
        times.append((time.perf_counter() - t0) * 1_000)
    synno_times[qid] = times

### Speedup

In [ ]:
total_synno = sum(sum(v) for v in synno_times.values())

print(f"{'Query':<8} {'DuckDB (ms)':>12} {'SynnoDB (ms)':>14} {'Speedup':>9}")
print("-" * 48)
for qid in spec.all_query_ids:
    d = sum(baseline_times[qid])
    s = sum(synno_times[qid])
    speedup = d / s if s > 0 else float("inf")
    print(f"Q{qid:<7} {d:>12.1f} {s:>14.1f} {speedup:>8.2f}x")
print("-" * 48)
overall = total_duck / total_synno if total_synno > 0 else float("inf")
print(f"{'TOTAL':<8} {total_duck:>12.1f} {total_synno:>14.1f} {overall:>8.2f}x")

### Correctness guarantee

Every routed result was compared against DuckDB (`cross_check_rate=1.0`).
The mismatch count must be 0.

In [ ]:
stats = con.router_stats()["session"]
print(f"Routed:         {stats['routed']}")
print(f"Cross-checked:  {stats['cross_checked']}")
print(f"Mismatches:     {stats['cross_check_mismatch']}")
assert stats["cross_check_mismatch"] == 0, "result divergence detected!"
print("\nAll results match DuckDB exactly.")

### Q1 result (with timing footer)

In [ ]:
_, q1_sql, _ = instantiations["1"][0]
con.execute(q1_sql)
con.show()
con.close()

---
## Where to go next

The base implementation is single-threaded. The same `SynnoDB` object carries each engine further:

```python
opt   = db.runOptimLoop(base_impl=impl)           # single-threaded SIMD / cache optimization
multi = db.addMultiThreading(optimized=opt)        # parallel execution across cores
rep   = db.checkSfCorrectness(source=multi, target_sf=50)  # correctness at larger SF
```

These chain **without W&B**: each stage restores the previous one's engine straight from
its git snapshot (`impl.snapshot_hash`, `opt.snapshot_hash`, ...) in the local workspace, so
the whole pipeline runs for non-W&B users out of the box. (This works within one session /
workspace; to chain across machines, log to W&B and pass the run id instead, e.g.
`db.runOptimLoop(base_impl_wandb_id=impl.run_id)`.)

> **Want W&B logging?** Pass `wandb_project="..."` (and/or `wandb_entity="..."`) to
> `SynnoDB(...)`. W&B is off unless one of them is set — nothing logs in, initializes, or
> requires credentials otherwise.

CLI equivalents and step-by-step commentary are in
[`docs/TUTORIAL_base_implementation.md`](../docs/TUTORIAL_base_implementation.md).